# XGBoost From Scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sindhug/xgboost-from-scratch/blob/main/xgboost_from_scratch.ipynb)
[![Watch the video](https://img.shields.io/badge/YouTube-Watch%20the%20video-red?logo=youtube)](https://youtu.be/Y0EJQFj0foo)

Companion notebook for **[XGBoost Explained: How the Algorithm Actually Works (Step-by-Step)](https://youtu.be/Y0EJQFj0foo)**.

We recompute, by hand, every number XGBoost would compute internally on the exact dataset from the video — a single feature (outdoor temperature) predicting a building's power consumption — then confirm it against the real `xgboost` library at the end.

Click **Open in Colab** above to run every cell live, no setup required.

## The dataset

Same 10 points used in the video: temperature (°C) vs. power consumed.

In [1]:
import numpy as np
import pandas as pd

temps = [0, 5, 10, 15, 20, 25, 30, 35, 40, 45]
power = [14, 11, 9, 7, 5, 5, 6, 8, 11, 12]

df = pd.DataFrame({"temp": temps, "power": power})
df

 temp  power
    0     14
    5     11
   10      9
   15      7
   20      5
   25      5
   30      6
   35      8
   40     11
   45     12

## Step 1 — Baseline: the mean

Gradient boosting always starts from a single number: the mean of the target.

In [2]:
baseline = df["power"].mean()
baseline

8.8

## Step 2 — Residuals & the root similarity score

Residuals are just "how far off is the baseline for each point." We also compute the
**similarity score** — sum of residuals, squared, divided by the count. Because positive
and negative residuals cancel out before squaring, the root's similarity is always 0 —
that's expected, not a bug.

In [3]:
df["resid"] = df["power"] - baseline
df

 temp  power  resid
    0     14    5.2
    5     11    2.2
   10      9    0.2
   15      7   -1.8
   20      5   -3.8
   25      5   -3.8
   30      6   -2.8
   35      8   -0.8
   40     11    2.2
   45     12    3.2

## The core formulas

These three functions are all of XGBoost's split-finding math for squared-error loss.
`lam` (λ, regularization) and `gamma` (γ, the minimum-gain bar) default to 0 for now —
we bring them back in the regularization section.

In [4]:
def leaf_value(residuals, lam=0.0):
    r = np.asarray(residuals, dtype=float)
    return r.sum() / (len(r) + lam)

def sse(residuals, lam=0.0):
    r = np.asarray(residuals, dtype=float)
    lv = leaf_value(r, lam)
    return float(((r - lv) ** 2).sum())

def similarity(residuals, lam=0.0):
    r = np.asarray(residuals, dtype=float)
    return float((r.sum() ** 2) / (len(r) + lam))

def gain(left, right, root, lam=0.0, gamma=0.0):
    return similarity(left, lam) + similarity(right, lam) - similarity(root, lam) - gamma

root_resid = df["resid"].values
print(f"SSE_root = {sse(root_resid):.2f}   Similarity_root = {similarity(root_resid):.2f}")

SSE_root = 87.60   Similarity_root = 0.00


## Step 3 — Finding the root split

XGBoost builds a special tree: it tests candidate thresholds, and for each one sends
residuals left/right, computes each side's leaf value (mean residual) and similarity,
then scores the split by **gain** = left similarity + right similarity − root similarity.
We test the three candidates from the video: 2.5, 7.5, 12.5.

In [5]:
for thr in [2.5, 7.5, 12.5]:
    left = df[df.temp <= thr]["resid"].values
    right = df[df.temp > thr]["resid"].values
    g = gain(left, right, root_resid)
    print(f"threshold={thr:>5} | leaf_L={leaf_value(left):+.4f} leaf_R={leaf_value(right):+.4f} "
          f"| SSE_L={sse(left):.4f} SSE_R={sse(right):.4f} | Gain={g:.4f}")

threshold=  2.5 | leaf_L=+5.2000 leaf_R=-0.5778 | SSE_L=0.0000 SSE_R=57.5556 | Gain=30.0444
threshold=  7.5 | leaf_L=+3.7000 leaf_R=-0.9250 | SSE_L=4.5000 SSE_R=48.8750 | Gain=34.2250
threshold= 12.5 | leaf_L=+2.5333 leaf_R=-1.0857 | SSE_L=12.6667 SSE_R=47.4286 | Gain=27.5048


**7.5 wins** with the largest gain (34.225), so it becomes the root split.

## Step 4 — Growing the tree one more level

We can split either child further. We compare:
- splitting the **left** child (`temp ≤ 7.5`, just 2 points) into two singletons
- splitting the **right** child (`temp > 7.5`, 8 points) at a new threshold, 37.5

...and keep whichever gives the bigger additional gain.

In [6]:
left_root  = df[df.temp <= 7.5]["resid"].values
right_root = df[df.temp > 7.5]["resid"].values

# Option A: split the left child into singletons
gain_left_split = sse(left_root) - 0.0
print(f"Gain from splitting left child into singletons: {gain_left_split:.4f}")

# Option B: split the right child at 37.5
right_left  = df[(df.temp > 7.5) & (df.temp <= 37.5)]["resid"].values
right_right = df[df.temp > 37.5]["resid"].values
gain_right_split = sse(right_root) - (sse(right_left) + sse(right_right))
print(f"Gain from splitting right child at 37.5:        {gain_right_split:.4f}")
print(f"\nleaf(7.5 < temp <= 37.5) = {leaf_value(right_left):.4f}")
print(f"leaf(temp > 37.5)        = {leaf_value(right_right):.4f}")

Gain from splitting left child into singletons: 4.5000
Gain from splitting right child at 37.5:        35.0417

leaf(7.5 < temp <= 37.5) = -2.1333
leaf(temp > 37.5)        = 2.7000


35.04 > 4.50, so we split the **right** child at 37.5 instead. The final depth-2 tree
has three leaves:

```
                 temp <= 7.5?
                /            \
             yes              no
              |                |
          leaf = +3.70    temp <= 37.5?
                          /            \
                       yes              no
                        |                |
                 leaf = -2.1333    leaf = +2.70
```

## Step 5 — Making a prediction: F(x)

A prediction is: start at the baseline, send `x` down the tree to a leaf, add that leaf's
correction. This is the **no-shrinkage** version — full-strength corrections.

In [7]:
def predict_tree1(x, eta=1.0):
    if x <= 7.5:
        corr = leaf_value(left_root)          # +3.70
    elif x <= 37.5:
        corr = leaf_value(right_left)         # -2.1333
    else:
        corr = leaf_value(right_right)        # +2.70
    return baseline + eta * corr

for x in [4, 33, 42]:
    print(f"x={x:>3} -> F(x) = {predict_tree1(x):.4f}")

x=  4 -> F(x) = 12.5000
x= 33 -> F(x) = 6.6667
x= 42 -> F(x) = 11.5000


## Step 6 — Learning rate (shrinkage, η)

η scales the leaf correction before it's added — a volume knob on how loudly each tree
speaks. It doesn't change *which* split gets picked (that's still decided by gain); it
only changes how much of that tree's correction survives into the running prediction —
which in turn changes how large next round's residuals stay.

In [8]:
for eta in [0.5, 0.3]:
    preds = [round(predict_tree1(x, eta), 4) for x in [4, 33, 42]]
    print(f"eta={eta}: {preds}")

eta=0.5: [10.65, 7.7333, 10.15]
eta=0.3: [9.91, 8.16, 9.61]


## Step 7 — Round-1 predictions & the next round's residuals (η = 0.5)

Every sample now gets a prediction from tree 1, shrunk by η=0.5. The **new residuals**
(true value − prediction) become the training target for tree 2 — gradient boosting
always fits the next tree to whatever the current model still gets wrong.

In [9]:
df["pred1"]  = df["temp"].apply(lambda x: predict_tree1(x, eta=0.5))
df["resid1"] = df["power"] - df["pred1"]
df

 temp  power  resid     pred1    resid1
    0     14    5.2 10.650000  3.350000
    5     11    2.2 10.650000  0.350000
   10      9    0.2  7.733333  1.266667
   15      7   -1.8  7.733333 -0.733333
   20      5   -3.8  7.733333 -2.733333
   25      5   -3.8  7.733333 -2.733333
   30      6   -2.8  7.733333 -1.733333
   35      8   -0.8  7.733333  0.266667
   40     11    2.2 10.150000  0.850000
   45     12    3.2 10.150000  1.850000

In [10]:
sse_before = sse(root_resid)
sse_after  = float((df["resid1"] ** 2).sum())
print(f"SSE at baseline    = {sse_before:.2f}")
print(f"SSE after 1 tree   = {sse_after:.2f}")

SSE at baseline    = 87.60
SSE after 1 tree   = 35.65


One tree, half-strength, and the error already dropped by more than half. Tree 2 would
now repeat the exact same recipe — test thresholds, pick the best split, grow a level —
but targeting `resid1` instead of the original residuals from the mean. Every later tree
follows the same pattern, always fitting whatever the current ensemble still misses.

## Tree capacity & stopping

A few knobs control how big any single tree — and the whole ensemble — is allowed to get:

- **`max_depth` / `max_leaves`** — caps how many times a tree can split. Smaller trees make
  gentler, more local corrections.
- **`gamma` (min_split_loss)** — a split only happens if its gain clears this bar. Below it,
  the gain is treated as not worth the extra leaf.
- **Early stopping** — stop adding trees once a validation metric stops improving, regardless
  of how many rounds were requested.

We can see `gamma` in action on the two candidate splits from Step 4:

In [11]:
for g in [0, 5, 10, 40]:
    print(f"gamma={g:>3} -> split left child?  {4.5 - g > 0}   |   split right child at 37.5?  {35.0417 - g > 0}")

gamma=  0 -> split left child?  True   |   split right child at 37.5?  True
gamma=  5 -> split left child?  False   |   split right child at 37.5?  True
gamma= 10 -> split left child?  False   |   split right child at 37.5?  True
gamma= 40 -> split left child?  False   |   split right child at 37.5?  False


At `gamma=40`, even the best split (35.04 gain) isn't worth it — the node stays a leaf.
That's the whole mechanism: gamma is a toll charged on every new leaf.

## Regularization (λ)

Adding λ to the denominator of similarity (and leaf value) is like adding λ "virtual,
zero-valued" points to every leaf. Small or inconsistent groups of residuals get
pulled toward a leaf value of 0, while large, consistent groups barely change. Watch what
happens to the 7.5-threshold split as λ increases:

In [12]:
left  = df[df.temp <= 7.5]["resid"].values
right = df[df.temp > 7.5]["resid"].values

for lam in [0, 1, 5, 20]:
    lv_l, lv_r = leaf_value(left, lam), leaf_value(right, lam)
    g = gain(left, right, root_resid, lam=lam)
    print(f"lambda={lam:>2} -> leaf_L={lv_l:+.4f}  leaf_R={lv_r:+.4f}  Gain={g:.4f}")

lambda= 0 -> leaf_L=+3.7000  leaf_R=-0.9250  Gain=34.2250
lambda= 1 -> leaf_L=+2.4667  leaf_R=-0.8222  Gain=25.8410
lambda= 5 -> leaf_L=+1.0571  leaf_R=-0.5692  Gain=11.6127
lambda=20 -> leaf_L=+0.3364  leaf_R=-0.2643  Gain=3.7773


As λ grows, both leaf values and the gain shrink toward zero — only strong, consistent
residuals stay attractive enough to justify a split. Combined with gamma charging a toll on
every new leaf, the two together keep trees modest and reduce overfitting on noisy splits.

## Verifying against the real `xgboost` library

Everything above was reimplemented by hand. Let's confirm it against the actual library —
one tree, no shrinkage-defeating extras, matching our exact setup (`max_depth=2`,
`learning_rate=0.5`, `n_estimators=1`, no regularization).

In [ ]:
!pip install -q xgboost

import xgboost as xgb

X = df[["temp"]].values
y = df["power"].values

model = xgb.XGBRegressor(
    n_estimators=1,
    max_depth=2,
    learning_rate=0.5,
    reg_lambda=0,
    gamma=0,
    base_score=baseline,
    objective="reg:squarederror",
)
model.fit(X, y)

xgb_preds = model.predict(X)
print("xgboost predictions:      ", np.round(xgb_preds, 4))
print("hand-computed predictions:", df["pred1"].round(4).values)

If the two rows match (small floating-point differences aside), our from-scratch math
lines up exactly with what XGBoost computes internally.

## Summary — what makes it "Extreme"

XGBoost is still gradient boosting: many small trees, each fixing what the current model
gets wrong. "Extreme" is about power, control, and scale:

- Uses both the error **and its curvature** (gradients and Hessians) to set leaf values and
  pick splits, so each tree targets the most important mistakes faster.
- Bakes in **regularization** (λ on leaf weights, γ on leaf count) to fight overfitting.
- Handles **missing/sparse data** natively.
- Heavily optimized (histogram-based splits, parallelism, GPU support, subsampling) so it
  trains fast on large datasets — even though, as this notebook shows, the core math works
  the same on 10 rows as it does on 10 million.

---

⭐ If this made XGBoost click, star [this repo](https://github.com/sindhug/xgboost-from-scratch) and watch the [full video](https://youtu.be/Y0EJQFj0foo).